# 04 - LLM Setup and Stratified Sampling

This notebook initializes the local zero-shot setup and creates a reusable 200-post sample for all later experiments.

In [1]:
import os
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
import sys
sys.modules["tensorflow"] = None

In [2]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

Found existing installation: torch 2.10.0
Uninstalling torch-2.10.0:
  Successfully uninstalled torch-2.10.0


Looking in indexes: https://download.pytorch.org/whl/cpu
  Obtaining dependency information for torch from https://download-r2.pytorch.org/whl/cpu/torch-2.11.0%2Bcpu-cp310-cp310-win_amd64.whl.metadata
  Obtaining dependency information for torchvision from https://download-r2.pytorch.org/whl/cpu/torchvision-0.26.0%2Bcpu-cp310-cp310-win_amd64.whl.metadata
  Obtaining dependency information for torchaudio from https://download-r2.pytorch.org/whl/cpu/torchaudio-2.11.0%2Bcpu-cp310-cp310-win_amd64.whl.metadata
   ---------------------------------------- 0.0/114.4 MB ? eta -:--:--
   ---------------------------------------- 0.2/114.4 MB 3.7 MB/s eta 0:00:31
   ---------------------------------------- 0.4/114.4 MB 4.5 MB/s eta 0:00:26
   ---------------------------------------- 0.9/114.4 MB 6.5 MB/s eta 0:00:18
    --------------------------------------- 1.8/114.4 MB 9.4 MB/s eta 0:00:12
    --------------------------------------- 2.4/114.4 MB 10.3 MB/s eta 0:00:11
    -----------------------


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: C:\Users\HP\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from transformers import pipeline

RANDOM_STATE = 42
root = Path.cwd()
outputs_dir = root / "outputs"
if not outputs_dir.exists():
    outputs_dir = root.parent / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

data_csv = outputs_dir / "reddit_cleaned_balanced.csv"
df = pd.read_csv(data_csv)
required_cols = {"text_clean", "risk_label"}
missing = required_cols - set(df.columns)
if missing:
    raise KeyError(f"Missing columns in {data_csv}: {missing}")

df = df.reset_index(drop=False).rename(columns={"index": "row_id"})
sample_df = (
    df.groupby("risk_label", group_keys=False)
      .apply(lambda g: g.sample(n=100, random_state=RANDOM_STATE, replace=False))
      .reset_index(drop=True)
)

sample_df.to_csv(outputs_dir / "llm_sample.csv", index=False)
np.save(outputs_dir / "llm_sample_row_ids.npy", sample_df["row_id"].astype(int).to_numpy())

try:
    classifier = pipeline(
        "zero-shot-classification",
        model="facebook/bart-large-mnli",
        device=-1,
    )
except Exception as e:
    raise RuntimeError(
        "Failed to load facebook/bart-large-mnli locally. "
        "Install transformers and torch, then rerun. "
        f"Original error: {e}"
    )

meta = {
    "sample_size": int(len(sample_df)),
    "class_counts": sample_df["risk_label"].value_counts().sort_index().to_dict(),
    "model": "facebook/bart-large-mnli",
    "device": -1,
    "random_state": RANDOM_STATE
}
(outputs_dir / "llm_setup_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print(meta)
sample_df.head(3)

C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\HP\AppData\Local\Temp\ipykernel_23448\2019424528.py:24: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=100, random_state=RANDOM_STATE, replace=False))
Loading weights: 100%|██████████| 515/515 [00:00<00:00, 1294.45it/s]


{'sample_size': 200, 'class_counts': {0: 100, 1: 100}, 'model': 'facebook/bart-large-mnli', 'device': -1, 'random_state': 42}


,row_id,risk_label,risk_label_name,subreddit,author,created_utc,timestamp,word_count,text_clean
0,1860,0,Mental Health Risk,mentalhealth,Klimpf,1598915474,2020-09-01 09:11:14,224,sometimes i (m24) just get so sentimental and ...
1,353,0,Mental Health Risk,anxiety,OkRecommendation4830,1659208428,2022-07-31 05:13:48,278,"i know there's no diagnosing on this sub, just..."
2,1333,0,Mental Health Risk,depression,luke4ario,1555589278,2019-04-18 22:07:58,251,i feel like nothing is worth it. i feel like t...
